In [0]:
%run ./01_setup_volumen

In [0]:
import polars as pl

silver_path = settings.silver / "orders" / f"run_date={settings.run_date}" / "part-000.parquet"
silver_df = pl.read_parquet(silver_path)

fact_sales = silver_df

agg_daily_sales = (
    silver_df
    .filter(pl.col("status") == "paid")
    .group_by(["order_date", "country", "category"])
    .agg([
        pl.len().alias("total_orders"),
        pl.sum("quantity").alias("total_units"),
        pl.sum("gross_amount").round(2).alias("total_revenue"),
        pl.mean("gross_amount").round(2).alias("avg_ticket"),
    ])
    .sort(["order_date", "country", "category"])
)

customer_ltv = (
    silver_df
    .filter(pl.col("status") == "paid")
    .group_by(["customer_id", "country"])
    .agg([
        pl.len().alias("paid_orders"),
        pl.sum("gross_amount").round(2).alias("lifetime_value"),
        pl.max("order_date").alias("last_order_date"),
    ])
    .sort("lifetime_value", descending=True)
)

fact_dir = settings.gold / "fact_sales" / f"run_date={settings.run_date}"
agg_dir = settings.gold / "agg_daily_sales" / f"run_date={settings.run_date}"
ltv_dir = settings.gold / "customer_ltv" / f"run_date={settings.run_date}"

for p in [fact_dir, agg_dir, ltv_dir]:
    p.mkdir(parents=True, exist_ok=True)

fact_sales.write_parquet(fact_dir / "part-000.parquet")
agg_daily_sales.write_parquet(agg_dir / "part-000.parquet")
customer_ltv.write_parquet(ltv_dir / "part-000.parquet")

print("Gold publicada en:", settings.gold)


In [0]:
fact_sales.head(5)

In [0]:
agg_daily_sales.head(5)

In [0]:
customer_ltv.head(5)